# Comparing Cohere models with `compare_models`: all 4 Cohere architectures

Two comparison runs, **Model A vs Model B**, together cover the four Cohere architectures AuditKIT supports on the `hf:` backend:

| Run | Model A | Model B | Task |
|---|---|---|---|
| **1 · text** | Tiny Aya Global (`cohere2`, 3.3B) | Aya Expanse 8B (`cohere`) | multilingual QA and translation |
| **2 · vision** | Aya Vision 8B (`aya_vision`) | North Micro Vision (`cohere_compass`) | questions about images (colour, counting, reading text, a chart) |

`ak.compare_models` runs the same samples and metrics on each model, **loading one model at a time** and freeing it before the next. It then reports:
- per-metric scores and the winner;
- a paired significance test;
- per-sample disagreements;
- latency, throughput and model size.

**Runtime:** a Colab **L4 or A100** (24 GB+). Aya Expanse and Aya Vision are 8B models (~16–17 GB in bf16). On a **T4**, set `LOAD_IN_4BIT = True`, which needs `bitsandbytes`.

**Access:**
- Tiny Aya, Aya Expanse and Aya Vision are **gated**: accept each licence on the Hub, then add an `HF_TOKEN` Colab secret.
- North Micro Vision is open.
- Add a `GITHUB_TOKEN` secret to install AuditKIT from the private repo.

In [ ]:
# ---- settings ----
import os
RUN1 = (os.environ.get("NB_RUN1_A", "CohereLabs/tiny-aya-global"),          # Model A: cohere2
        os.environ.get("NB_RUN1_B", "CohereLabs/aya-expanse-8b"))           # Model B: cohere
RUN2 = (os.environ.get("NB_RUN2_A", "CohereLabs/aya-vision-8b"),            # Model A: aya_vision
        os.environ.get("NB_RUN2_B", "CohereLabs/North-Micro-Vision-Instruct"))  # Model B: cohere_compass
LOAD_IN_4BIT = False            # True on a 16 GB T4 (pip install bitsandbytes)
MAX_TOKENS = 64
RUN_TEXT = os.environ.get("NB_SKIP_TEXT") != "1"
RUN_VISION = os.environ.get("NB_SKIP_VISION") != "1"

In [ ]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at REPO_BRANCH and installs it with the extras this notebook needs.
# Local: uses the auditkit already importable (e.g. the repo's .venv kernel).
import os, sys, subprocess
REPO_BRANCH = "feat/rag-agent-evals"   # #1's branch: #9, #10 and every #15 / #27 fix are merged here
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ("GITHUB_TOKEN", "HF_TOKEN"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", REPO_BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[transformers,vision]"], check=True)
    sys.path.insert(0, "/content/AuditKIT-Internal/src")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
else:
    print("HF_TOKEN not set: gated Cohere models (Tiny Aya, Aya Expanse, Aya Vision) will fail to download.")

In [ ]:
import auditkit as ak, torch, transformers
from auditkit.runspec import RunConfig
from auditkit.model.hf_gen import HFGenModel
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if (BF16 or DEVICE == "mps") else torch.float16
print(f"auditkit {ak.__version__} | transformers {transformers.__version__} | torch {torch.__version__}")
print(f"device {DEVICE} | GPU {GPU_GB:.0f} GB | dtype {DTYPE}")
assert int(transformers.__version__.split(".")[0]) >= 5, "the Cohere models need transformers >= 5.15"

In [ ]:
def hf(name):
    """An hf: model with the right dtype for this GPU (and 4-bit on a T4 if asked)."""
    kw = {"dtype": DTYPE}
    if LOAD_IN_4BIT:
        kw = {"load_in_4bit": True}
    if os.environ.get("HF_TOKEN"):
        kw["token"] = os.environ["HF_TOKEN"]
    return HFGenModel(name, device=DEVICE, name=name.split("/")[-1], **kw)

def show(cmp, samples):
    print(cmp.summary())
    print("\nper metric:")
    for row in cmp.comparison_table():
        print("  ", row)
    names = list(cmp.runs)
    if len(names) == 2:
        for metric in [m for m in cmp.runs[names[0]].headline if m in cmp.runs[names[1]].headline]:
            sig = cmp.significance(names[0], names[1], metric)
            print(f"  significance [{metric}]: {sig}")
    print("\nperformance:")
    for row in cmp.performance_table():
        print("  ", row)
    print("\nanswers side by side:")
    outs = {n: {p.sample_id: (p.raw_output or "").strip().replace("\n", " ")[:70] for p in r.predictions}
            for n, r in cmp.runs.items()}
    for s in samples:
        print(f"  [{s.id}] target={s.target!r}")
        for n in names:
            print(f"      {n:28s} {outs[n].get(s.id)!r}")

## Run 1 · text: Tiny Aya (`cohere2`) vs Aya Expanse (`cohere`)

Multilingual questions and translations in French, Spanish, German, Hindi, Arabic, Japanese, Turkish and Swahili. The Aya models are built for this. Each prompt asks for a one-word or one-number answer, so the lexical metrics are meaningful:
- `quasi_exact_match`: normalised exact match;
- `f1_score`: token overlap;
- `chrf`: character n-grams, which is fair to morphologically rich languages.

In [ ]:
from auditkit import Sample
TEXT = [
    ("fr-translate", "Translate to French. Reply with the translation only: good morning", "bonjour"),
    ("es-translate", "Translate to Spanish. Reply with the translation only: thank you", "gracias"),
    ("de-math", "Was ist 7 mal 8? Antworte nur mit der Zahl.", "56"),
    ("hi-capital", "भारत की राजधानी क्या है? केवल नाम लिखें।", "नई दिल्ली"),
    ("ar-capital", "ما هي عاصمة مصر؟ أجب بكلمة واحدة فقط.", "القاهرة"),
    ("ja-capital", "日本の首都はどこですか？一語で答えてください。", "東京"),
    ("tr-color", "Gökyüzü ne renktir? Tek kelimeyle cevap ver.", "mavi"),
    ("sw-number", "Tano jumlisha tatu ni ngapi? Jibu kwa namba tu.", "8"),
    ("en-capital", "What is the capital of Australia? Answer with one word.", "Canberra"),
    ("fr-capital", "Quelle est la capitale de l'Italie ? Réponds en un mot.", "Rome"),
]
text_samples = [Sample(id=i, input=q, target=a) for i, q, a in TEXT]
TEXT_METRICS = ["quasi_exact_match", "f1_score", "chrf"]
print(len(text_samples), "samples")

In [ ]:
if RUN_TEXT:
    cmp1 = ak.compare_models(
        [hf(RUN1[0]), hf(RUN1[1])], text_samples, TEXT_METRICS,
        model_names=[f"A: {RUN1[0].split('/')[-1]}", f"B: {RUN1[1].split('/')[-1]}"],
        config=RunConfig(max_tokens=MAX_TOKENS, temperature=0.0),
    )
    show(cmp1, text_samples)
    print("\nwinner on chrf:", cmp1.winner("chrf"))

## Run 2 · vision: Aya Vision (`aya_vision`) vs North Micro Vision (`cohere_compass`)

The images are generated here with Pillow, so the notebook needs no downloads and every answer is known: a coloured shape, a count, printed text, and a bar chart. On the `hf:` backend, `Sample.images` goes through each model's own processor and chat template. A text-only model would refuse images with a clear error, rather than silently ignoring them.

The replies to image questions are often sentences ("The square is red."), so besides `f1_score` this run uses a small **custom metric**: *does the reply mention the target?*

In [ ]:
from PIL import Image, ImageDraw, ImageFont

def canvas():
    return Image.new("RGB", (336, 336), "white")

def red_square():
    im = canvas(); ImageDraw.Draw(im).rectangle([88, 88, 248, 248], fill=(220, 30, 30)); return im

def three_circles():
    im = canvas(); d = ImageDraw.Draw(im)
    for x in (60, 168, 276):
        d.ellipse([x - 40, 128, x + 40, 208], fill=(30, 60, 220))
    return im

def printed_number():
    im = canvas(); d = ImageDraw.Draw(im)
    try:
        font = ImageFont.load_default(size=96)
    except TypeError:           # Pillow < 10.1
        font = ImageFont.load_default()
    d.text((70, 110), "42", fill="black", font=font); return im

def bar_chart():
    im = canvas(); d = ImageDraw.Draw(im)
    for i, (label, h) in enumerate([("A", 90), ("B", 150), ("C", 260), ("D", 120)]):
        x = 40 + i * 70
        d.rectangle([x, 300 - h, x + 45, 300], fill=(40, 140, 80))
        d.text((x + 18, 306), label, fill="black")
    return im

def green_triangle():
    im = canvas(); ImageDraw.Draw(im).polygon([(168, 60), (60, 270), (276, 270)], fill=(30, 170, 60)); return im

VISION = [
    ("color", red_square(), "What color is the shape in this image? Answer with one word.", "red"),
    ("count", three_circles(), "How many circles are in this image? Answer with a number.", "3"),
    ("read", printed_number(), "What number is written in this image? Answer with the number only.", "42"),
    ("chart", bar_chart(), "This is a bar chart with bars labelled A, B, C and D. Which bar is the tallest? Answer with the letter.", "C"),
    ("shape", green_triangle(), "What shape is shown in this image? Answer with one word.", "triangle"),
]
vision_samples = [Sample(id=i, input=q, target=a, images=[img]) for i, img, q, a in VISION]
try:
    from IPython.display import display
    display(*[img.resize((112, 112)) for _, img, _, _ in VISION])
except ImportError:
    pass

In [ ]:
from auditkit.metric import Metric
from auditkit.score import Score
from auditkit.types import Direction, ScoreKind

class MentionsTarget(Metric):
    """1.0 when the reply mentions the target as a whole word (case-insensitive)."""
    name = "mentions_target"
    kind = ScoreKind.BENCHMARK
    direction = Direction.MAXIMIZE

    def score(self, sample, output, context=None):
        import re
        hit = re.search(rf"(?<![\w]){re.escape(str(sample.target).lower())}(?![\w])", (output or "").lower())
        return Score(name=self.name, value=1.0 if hit else 0.0, kind=self.kind)

if RUN_VISION:
    cmp2 = ak.compare_models(
        [hf(RUN2[0]), hf(RUN2[1])], vision_samples, [MentionsTarget(), "f1_score"],
        model_names=[f"A: {RUN2[0].split('/')[-1]}", f"B: {RUN2[1].split('/')[-1]}"],
        config=RunConfig(max_tokens=MAX_TOKENS, temperature=0.0),
    )
    show(cmp2, vision_samples)

## Reading the results
- **`summary()` / `comparison_table()`**: per-metric scores for A and B, with the difference.
- **`significance()`**: a paired test over the same samples. With only 5–10 samples, treat it as a sanity check rather than proof.
- **`performance_table()`**: latency and throughput per model on this GPU.
- **Side by side**: where A and B disagree. This is the fastest way to see *why* one model wins.

**Pushing results:** `cmp.push_to_hub("<you>/aya-comparison")` uploads both runs, with provenance, to a Hub dataset.